# Adaptive Recursive RAG with Binary Evidence Verification

This notebook implements and benchmarks three RAG architectures:
1. **Standard RAG**: Retrieves once and generates.
2. **Always Recursive RAG**: Forces exactly 3 retrieval loops.
3. **Adaptive Recursive RAG**: Retrieves, extracts claims, and verifies against context. Dynamically refines the query and loops up to 2 times only if necessary. Can abstain safely.

---

## 1. Setup & Imports
Installing and importing necessary libraries (`faiss-cpu`, `bitsandbytes`, `transformers`, `datasets`, `sentence-transformers`, `matplotlib`, `pandas`).

In [ ]:
!pip install -q \
transformers==4.41.2 \
accelerate==0.30.1 \
bitsandbytes==0.43.1 \
sentence-transformers==2.7.0 \
faiss-cpu \
datasets \
pandas \
matplotlib

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import torch

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


## 2. Dataset Loading & Corpus Construction

In [ ]:
print("Loading HotpotQA (distractor) validation split, sample 75...")
dataset = load_dataset("hotpot_qa", "distractor", split="validation")
sample = dataset.select(range(75))

print("Constructing corpus with deduplication...")
all_passages = []
seen = set()

for example in sample:
    titles = example["context"]["title"]
    sentences_list = example["context"]["sentences"]
    for title, sentences in zip(titles, sentences_list):
        passage = title + ". " + " ".join(sentences)
        if passage not in seen:
            seen.add(passage)
            all_passages.append(passage)

print(f"Sample size: {len(sample)} questions.")
print(f"Corpus size: {len(all_passages)} unique passages built for retrieval.")

## 3. Embedding Model & Vector Index (FAISS)

In [ ]:
print("Loading BGE-small-en-v1.5 embedding model...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("Embedding passages...")
passage_embeddings = embed_model.encode(
    all_passages,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

print("Building FAISS IndexFlatIP...")
dimension = passage_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)
index.add(passage_embeddings)
print(f"Index built with {index.ntotal} vectors.")

def embed_query(query):
    prefixed_query = "Represent this sentence for searching relevant passages: " + query
    return embed_model.encode([prefixed_query], normalize_embeddings=True)

def retrieve(query, existing_docs, k=3):
    query_emb = embed_query(query)
    # query_emb is already normalized
    scores, indices = index.search(query_emb, k)
    new_docs = [all_passages[i] for i in indices[0]]
    for doc in new_docs:
        if doc not in existing_docs:
            existing_docs.append(doc)
    return existing_docs

## 4. Language Model & Core Functions (generation, extraction, verification)

In [ ]:
print("Loading Phi-3 Mini in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_id = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"
)

llm.eval()

def llm_call(system_message, user_message, max_new_tokens):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(llm.device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.1
        )

    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def generate(query, docs):
    system_msg = (
        "You are a factual question answering assistant.\n"
        "You will be given a question and a set of reference documents.\n"
        "Answer the question using only the information in the documents.\n"
        "Be concise. Do not add information that is not in the documents.\n"
        "If the documents do not contain enough information, say:\n"
        "\"I cannot determine this from the provided documents.\""
    )

    docs_text = "\n\n".join(docs)
    user_msg = f"Documents:\n{docs_text}\n\nQuestion: {query}\n\nAnswer:"
    return llm_call(system_msg, user_msg, max_new_tokens=256)

ABSTAIN_PHRASES = [
    "cannot determine", "cannot find", "not enough information", 
    "insufficient information", "not provided in the documents", 
    "not mentioned in the documents", "i don\t know", 
    "unable to determine", "no information", "not stated in"
]

def is_abstain_answer(answer):
    ans_lower = answer.lower().strip()
    return any(phrase in ans_lower for phrase in ABSTAIN_PHRASES)

def extract_claims(answer):
    system_msg = (
        "You are a precise factual analyzer.\n"
        "Your job is to break a given answer into individual factual claims.\n"
        "Each claim must contain exactly ONE fact — do not combine multiple facts into one claim.\n"
        "Each claim must be a single standalone statement that can be\n"
        "verified independently.\n"
        "Output exactly 2 to 3 claims, one per line, numbered.\n"
        "Do not output anything else — no explanation, no preamble,\n"
        "no extra text."
    )

    user_msg = f"Answer: {answer}\n\nBreak this answer into 2 to 3 factual claims, one per line, numbered:\n1."
    raw_output = llm_call(system_msg, user_msg, max_new_tokens=128)
    raw_output = "1. " + raw_output

    import re
    claims = []
    lines = raw_output.split("\n")
    for line in lines:
        cleaned = line.strip()
        if not cleaned: continue
        cleaned = re.sub(r"^\d+[\.\)\-]\s*", "", cleaned)
        cleaned = re.sub(r"^[\-\*]\s*", "", cleaned)
        cleaned = re.sub(r"\s*\(Claim\s*#?\d+\)\s*$", "", cleaned, flags=re.IGNORECASE).strip()

        if cleaned and len(cleaned) > 15:
            claims.append(cleaned)

    if len(claims) > 3:
        claims = claims[:3]

    if len(claims) < 2:
        sentences = re.split(r"[\.\?\!]", answer)
        claims = [s.strip() for s in sentences if len(s.strip()) > 15][:3]

    return claims


def verify(claims, docs):
    system_msg = (
        "You are a strict fact verification assistant.\n"
        "You will be given a factual claim and a set of reference documents.\n"
        "Your job is to determine if the claim is supported by the documents.\n"
        "Reply with exactly one word: Yes or No.\n"
        "Do not explain. Do not add any other text. Just Yes or No."
    )

    docs_text = "\n\n".join(docs)
    results = []
    for claim in claims:
        user_msg = f"Documents:\n{docs_text}\n\nClaim: {claim}\n\nIs this claim supported by the documents above? Answer Yes or No:"
        raw_output = llm_call(system_msg, user_msg, max_new_tokens=8).lower().strip()
        res = False
        if raw_output.startswith("yes"):
            res = True
        elif raw_output.startswith("no"):
            res = False
        elif "yes" in raw_output[:20]:
            res = True
        elif "no" in raw_output[:20]:
            res = False
        results.append(res)
    return results

def refine_query(original_query, failed_claim):
    system_msg = "You are an intelligent search assistant."
    user_msg = f"Generate a concise search query to retrieve factual information.\nKeep key entities from the original question.\n\nOriginal question: {original_query}\nClaim: {failed_claim}\n\nSearch query:"
    raw = llm_call(system_msg, user_msg, max_new_tokens=32)
    return raw.strip()

def filter_claims(claims, answer):
    keywords = set(answer.lower().split())
    return [c for c in claims if any(k in c.lower() for k in keywords)]

def normalize(text):
    return "".join(c.lower() for c in text if c.isalnum() or c.isspace()).strip()


## 5. Pipeline Orchestrator (The Three Modes)

In [ ]:
def check_correctness(answer, ground_truth):
    if not isinstance(answer, str) or not isinstance(ground_truth, str):
        return False
    ans_clean = normalize(answer)
    gt_clean = normalize(ground_truth)
    return gt_clean in ans_clean

def run_pipeline(query, ground_truth, mode="standard"):
    start_time = time.time()
    result = {
        "answer": "",
        "steps": 0,
        "verified": False,
        "abstained": False,
        "mode": mode,
        "correct": False,
        "latency": 0.0
    }
    
    docs = []
    
    if mode == "standard":
        docs = retrieve(query, docs, k=8)
        answer = generate(query, docs)
        result["answer"] = answer
        result["steps"] = 1
        
    elif mode == "recursive":
        k_schedule = [8, 9, 10]
        for loop_idx in range(3):
            docs = retrieve(query, docs, k=k_schedule[loop_idx])
            answer = generate(query, docs)
        result["answer"] = answer
        result["steps"] = 3
        
    elif mode == "adaptive":
        current_query = query
        MAX_DEPTH = 2
        for step in range(1, MAX_DEPTH + 1):
            k_val = 8 + (step - 1)
            docs = retrieve(current_query, docs, k=k_val)
            answer = generate(query, docs) # Generate uses original query
            result["steps"] = step
            
            raw_claims = extract_claims(answer)
            claims = filter_claims(raw_claims, answer)
            if not claims: 
                claims = raw_claims
            
            verifications = verify(claims, docs)
            threshold = max(1, len(verifications) // 2)
            verified_sum = sum(verifications)
            
            if is_abstain_answer(answer) or verified_sum < threshold:
                if step == MAX_DEPTH:
                    result["abstained"] = True
                    result["answer"] = "Cannot find sufficient information." if is_abstain_answer(answer) else ""
                    break
                else:
                    if is_abstain_answer(answer):
                        current_query = query
                    else:
                        failed_claims = [c for c, v in zip(claims, verifications) if not v]
                        if failed_claims:
                            current_query = refine_query(query, failed_claims[0])
                        else:
                            current_query = query
                    continue
            else:
                result["verified"] = True
                result["answer"] = answer
                break
                    
    result["latency"] = round(time.time() - start_time, 2)
    if not result["abstained"]:
        result["correct"] = check_correctness(result["answer"], ground_truth)
    else:
        result["correct"] = False
        
    return result


## 6. Demo Execution

In [ ]:
# Demo Cell - Trace execution manually for verbosity

def match_query_to_dataset(target_query, sample):
    queries = [ex["question"] for ex in sample]
    q_emb = embed_model.encode([target_query], normalize_embeddings=True)
    db_emb = embed_model.encode(queries, normalize_embeddings=True)
    scores = np.dot(db_emb, q_emb.T).squeeze()
    idx = np.argmax(scores)
    return sample[idx]

target_query = "What WB supernatrual drama series did Rose Mcgowan star in?"
match = match_query_to_dataset(target_query, sample)

demo_q = match["question"]
demo_gt = match["answer"]

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"QUERY: {demo_q}")
print(f"GROUND TRUTH: {demo_gt}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")

# STANDARD MODE
print("MODE: standard")
docs = retrieve(demo_q, [], k=8)
ans = generate(demo_q, docs)
print(f"  Retrieved docs: {len(docs)}")
print(f"  Answer: \"{ans}\"")
print(f"  Steps: 1")
print(f"  Verified: False (not checked)")
print(f"  Abstained: False\n")

# RECURSIVE MODE
print("MODE: recursive")
docs = []
k_schedule = [8, 9, 10]
for idx in range(3):
    docs = retrieve(demo_q, docs, k=k_schedule[idx])
    ans = generate(demo_q, docs)
print(f"  Retrieved docs: {len(docs)}")
print(f"  Answer: \"{ans}\"")
print(f"  Steps: 3")
print(f"  Verified: False (not checked)")
print(f"  Abstained: False\n")

# ADAPTIVE MODE
print("MODE: adaptive")
docs = []
current_query = demo_q
MAX_DEPTH = 2
final_ans = ""
final_step = 0
abstained = False

for step in range(1, MAX_DEPTH + 1):
    k_val = 8 + (step - 1)
    docs = retrieve(current_query, docs, k=k_val)
    ans = generate(demo_q, docs)
    final_ans = ans
    final_step = step
    
    print(f"  --- Step {step} ---")
    if current_query != demo_q:
        print(f"  [Refined Search Query]: {current_query}")
        
    print(f"  Retrieved docs so far: {len(docs)}")
    print(f"  Generated Answer: \"{ans}\"")
    
    if is_abstain_answer(ans):
        print("  -> Model abstained directly.")
        if step == MAX_DEPTH:
            abstained = True
            final_ans = "Cannot find sufficient information."
            break
        else:
            current_query = demo_q
            print("  -> Getting more docs using original query...")
            continue
            
    raw_claims = extract_claims(ans)
    claims = filter_claims(raw_claims, ans)
    if not claims: 
        claims = raw_claims
        
    verifs = verify(claims, docs)
    print(f"  Extracted & Filtered Claims:")
    for c, v in zip(claims, verifs):
        print(f"    - {c} -> Supported: {v}")
        
    threshold = max(1, len(verifs) // 2)
    verified_sum = sum(verifs)
    
    if verified_sum >= threshold:
        print(f"  -> {verified_sum}/{len(verifs)} claims supported. Majority threshold met! Stopping early.")
        break
    else:
        if step == MAX_DEPTH:
            print(f"  -> Majority not met ({verified_sum}/{len(verifs)}). Max depth reached. Abstaining.")
            abstained = True
        else:
            failed = [c for c, v in zip(claims, verifs) if not v]
            if failed:
                print(f"  -> Majority not met. Refining query based on failed claim: \"{failed[0]}\"")
                current_query = refine_query(demo_q, failed[0])

print("\n  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  ADAPTIVE FINAL SUMMARY")
print(f"  Final Answer : {final_ans}")
print(f"  Total Steps  : {final_step}")
print(f"  Abstained    : {abstained}")
print(f"  Verified     : {not abstained}")
ans_clean = normalize(final_ans)
gt_clean = normalize(demo_gt)
correct = gt_clean in ans_clean
print(f"  Correct      : {correct}")
print(f"  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


## 7. Full Benchmark Run

## 8. Results & Visualization